# 18 · Calibration Workflows

Finstack ships a calibration helper that ingests market quotes, solves curves, and returns enriched `MarketContext` objects. This notebook wires the test quote set into `SimpleCalibration`.

### Learning Objectives
- Load reusable quote sets (rates, forwards, credit, vols)
- Configure `SimpleCalibration` with solver options
- Inspect calibration reports and derived market data

In [1]:
from datetime import date, timedelta
from pathlib import Path
import sys

from finstack.core.dates.schedule import Frequency
from finstack.core.market_data import MarketContext
from finstack.valuations import calibration as cal

project_root = Path.cwd()
helper_dir = project_root / "finstack-py" / "examples" / "scripts" / "valuations" / "calibration"
if str(helper_dir) not in sys.path:
    sys.path.append(str(helper_dir))
import calibration_capabilities as calib_helpers

base_date = date(2024, 1, 2)
market_quotes, forward_inputs, credit_inputs, swaption_specs = calib_helpers.build_market_quotes(base_date)

ModuleNotFoundError: No module named 'calibration_capabilities'

## 1. Discount Curve Calibration
We start by calibrating a USD-OIS discount curve directly with `DiscountCurveCalibrator`. The helper returns market quotes, so we recreate the short deposit and swap points inline before bootstrapping.

In [ ]:
discount_quotes = [
    cal.RatesQuote.deposit(base_date + timedelta(days=30), 0.0450, "ACT/360"),
    cal.RatesQuote.deposit(base_date + timedelta(days=90), 0.0465, "ACT/360"),
    cal.RatesQuote.swap(
        base_date + timedelta(days=365),
        0.0475,
        Frequency.ANNUAL,
        Frequency.QUARTERLY,
        "30/360",
        "ACT/360",
        "USD-SOFR",
    ),
    cal.RatesQuote.swap(
        base_date + timedelta(days=365 * 3),
        0.0485,
        Frequency.ANNUAL,
        Frequency.QUARTERLY,
        "30/360",
        "ACT/360",
        "USD-SOFR",
    ),
]

market = MarketContext()
discount_calibrator = cal.DiscountCurveCalibrator("USD-OIS", base_date, "USD")
discount_calibrator = discount_calibrator.with_config(
    cal.CalibrationConfig.multi_curve()
    .with_solver_kind(cal.SolverKind.HYBRID)
    .with_max_iterations(100)
)
discount_curve, discount_report = discount_calibrator.calibrate(discount_quotes)
market.insert_discount(discount_curve)
print("Discount curve calibrated:", discount_curve.id)
print("Report success:", discount_report.success)
print("Number of knots:", len(discount_curve.points))

In [ ]:
## 2. Forward, Credit, and Volatility Calibrations
With the discount curve loaded into a `MarketContext`, reuse the helper utilities to calibrate tenor-specific forwards, hazard curves, and populate swaption vol surfaces. These helpers are thin wrappers around the public calibrator types.

In [ ]:
forward_reports = calib_helpers.calibrate_forward_curves(market, base_date, forward_inputs)
credit_reports = calib_helpers.calibrate_credit_index_structures(market, base_date, credit_inputs)
fallback_surface = calib_helpers.ensure_swaption_surface(market, base_date, swaption_specs)
print("Forward curves calibrated:", list(forward_reports.keys()))
print("Credit objects calibrated:", list(credit_reports.keys()))
print("Swaption surface populated?", bool(fallback_surface))

In [ ]:
## 3. Market Snapshot
Inspect the aggregated context to confirm curve counts and metadata.

## 2. Run Calibration
Feed the quote set into the solver and then calibrate forward curves, credit indices, and swaption surfaces using helper routines.

In [ ]:
market, report = calibration.calibrate(market_quotes)
forward_reports = calib_helpers.calibrate_forward_curves(market, base_date, forward_inputs)
credit_reports = calib_helpers.calibrate_credit_index_structures(market, base_date, credit_inputs)
fallback_surface = calib_helpers.ensure_swaption_surface(market, base_date, swaption_specs)
report_info = report.to_dict()
print("Success:", report_info["success"])
print("Iterations:", report_info["iterations"])
print("Max residual:", report_info["max_residual"])
print("Convergence reason:", report_info["convergence_reason"])


## 3. Summarize Results
Display calibrated curve IDs, forward/hazard reports, and sample vol data via the helper `summarize_context`.

In [ ]:
calib_helpers.summarize_context(market, forward_reports, credit_reports)
if fallback_surface:
    print("Swaption surface bootstrapped from sample ATM grid")


## Summary
- `calibration_capabilities.build_market_quotes` bundles representative USD quotes across rates, credit, and vol.
- `SimpleCalibration` plus solver config calibrates discount/forward curves, credit spreads, and fills volatility surfaces.
- Helper utilities summarize the resulting `MarketContext`, ready for downstream pricing/risk.